In [ ]:
import gc
import torch
import time

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
import yaml
import torch

In [ ]:
import torch
import os, json, time, cv2, numpy as np
import pandas as pd
import gc
import random
import glob
from datetime import datetime
from ultralytics import YOLO

# --- CONFIGURATION ---
# (MUST MATCH DETECTRON2 CONFIG FOR APPLE-TO-APPLE COMPARISON)
VERSION = "C_2026_1d80_10_10_AUG"
DATASET_ROOT = f'/content/drive/MyDrive/Colab Notebooks/TA_CuttingRockDescription/datasets/{VERSION}'
DATASET_YAML = f"{DATASET_ROOT}/data.yaml"

TARGET_EPOCHS = 5
# TARGET_EPOCHS = 150
IMG_SIZE = 960
BATCH_SIZE = 6  # YOLO is more memory efficient, usually can handle 2x Detectron batch
PATIENCE = 50
RUNNER_NAME = "YOLOv12_Medium"
SINGLE_CLASS = False
# YOLOv12m model - will be downloaded from official release
YOLO_MODEL_URL = "https://github.com/sunsmarterjie/yolov12/releases/download/seg/yolov12m-seg.pt"
YOLO_MODEL = "yolov12m-seg.pt"
SMOKE_TEST = False

# Store VRAM usage per epoch (populated by callback)
vram_history = {}

def clear_cuda_cache():
    """Clear CUDA GPU memory cache."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()

def on_train_epoch_end(trainer):
    """Callback to log VRAM usage at the end of each epoch."""
    epoch = trainer.epoch + 1
    if torch.cuda.is_available():
        vram_gb = torch.cuda.memory_reserved() / 1E9
    else:
        vram_gb = 0.0
    vram_history[epoch] = vram_gb

def validate_smoke_test(output_dir, initial_loss=None, final_loss=None):
    """
    Validate YOLO smoke test.
    """
    checks = {
        "weights_saved": False,
        "csv_generated": False,
        "loss_decreased": False,
        "gpu_utilized": False,
    }
    details = {}

    # 1. Check Weights
    weights_dir = os.path.join(output_dir, "weights")
    if os.path.exists(os.path.join(weights_dir, "best.pt")) or os.path.exists(os.path.join(weights_dir, "last.pt")):
        checks["weights_saved"] = True

    # 2. Check CSV
    csv_path = os.path.join(output_dir, "results.csv")
    checks["csv_generated"] = os.path.exists(csv_path)

    # 3. Check Loss
    if initial_loss is not None and final_loss is not None:
        checks["loss_decreased"] = final_loss < initial_loss
        details["loss_delta"] = initial_loss - final_loss

    # 4. GPU
    if torch.cuda.is_available():
        checks["gpu_utilized"] = torch.cuda.max_memory_allocated() > 0
        details["max_vram"] = torch.cuda.max_memory_allocated() / 1E9

    print("\n" + "=" * 60)
    print("🔬 YOLO SMOKE TEST REPORT")
    print("=" * 60)
    for k, v in checks.items():
        print(f"  {'✅' if v else '❌'} {k}")
    print(f"  Details: {details}")
    print("=" * 60 + "\n")

def save_visual_predictions(model, dataset_yaml, output_dir, num_samples=3):
    """
    Generate visual predictions side-by-side (Raw vs Prediction).
    """
    print("Saving visual predictions...")
    vis_dir = os.path.join(output_dir, "visual_predictions")
    os.makedirs(vis_dir, exist_ok=True)

    # Load test image paths from yaml (parsing simplisticly)
    with open(dataset_yaml, 'r') as f:
        data_cfg = yaml.safe_load(f)

    test_path = data_cfg.get('test', data_cfg.get('val')) # Fallback to val if test not set
    # Handle if path is relative or absolute
    if not os.path.isabs(test_path):
        # Assuming test_path is relative to the directory containing data.yaml?
        # Ultralytics paths can be tricky, usually defined in settings.
        # For simplicity, let's construct it assuming standard structure:
        base_dir = os.path.dirname(dataset_yaml)
        test_images_dir = os.path.join(base_dir, "test", "images")
        if not os.path.exists(test_images_dir):
             test_images_dir = os.path.join(base_dir, test_path) # Try direct

    # Find images
    images = glob.glob(os.path.join(test_images_dir, "*.jpg")) + glob.glob(os.path.join(test_images_dir, "*.png"))

    if not images:
        print("⚠️ No images found for visualization.")
        return

    samples = random.sample(images, min(len(images), num_samples))

    for i, img_path in enumerate(samples):
        # Run inference
        results = model.predict(img_path, imgsz=IMG_SIZE, conf=0.25, save=False)

        # Plot result (returns numpy array)
        res_plotted = results[0].plot()

        # Save
        cv2.imwrite(os.path.join(vis_dir, f"pred_{i}.jpg"), res_plotted)

def download_model_if_needed(model_path, model_url):
    """Download model from URL if not already present."""
    if not os.path.exists(model_path):
        print(f"Downloading model from {model_url}...")
        import urllib.request
        urllib.request.urlretrieve(model_url, model_path)
        print(f"Model downloaded to {model_path}")
    else:
        print(f"Model already exists: {model_path}")

def run_yolo():
    # 1. Config Setup
    OUTPUT_DIR = f"{DATASET_ROOT}/models/{RUNNER_NAME}"

    # Download YOLOv12m if not present
    download_model_if_needed(YOLO_MODEL, YOLO_MODEL_URL)

    # model = YOLO(YOLO_MODEL)
    model = YOLO("/content/drive/MyDrive/Colab Notebooks/TA_CuttingRockDescription/datasets/C_2026_1d80_10_10_AUG/models/YOLOv12_Medium/weights/best.pt")

    # Add Custom Callback for VRAM logging
    model.add_callback("on_train_epoch_end", on_train_epoch_end)

    epochs = TARGET_EPOCHS
    batch = BATCH_SIZE

    if SMOKE_TEST:
        print("--- SMOKE TEST MODE ACTIVATED ---")
        epochs = 2 # Need at least 2 to see loss curve
        batch = 2
        patience = 0
    else:
        patience = PATIENCE

    clear_cuda_cache()

    # 2. Training
    print(f"Starting Training: {RUNNER_NAME}")
    results = model.train(
        data=DATASET_YAML,
        project=os.path.dirname(OUTPUT_DIR), # Ultralytics creates the subdir
        name=os.path.basename(OUTPUT_DIR),
        epochs=epochs,
        imgsz=IMG_SIZE,
        batch=batch,
        patience=patience,
        single_cls=SINGLE_CLASS,
        exist_ok=True, # Overwrite existing
        device=0 if torch.cuda.is_available() else 'cpu',
        val=True, # Validate every epoch,

        # Hyperparameter
        amp = True,
        retina_masks=True,   # High-res validation masks
        # overlap_mask=True,   # Crucial for touching/dense grains
        # mask_ratio=2,        # Prevents 4x downsampling of mask features
        # box=8,             # Optional: slightly boost box gain if detection is missed
        # mask=15.0            # Boost mask loss weight (default is 12.0)
    )

    # 3. Rich Log Generation (Parsing CSV + Merging VRAM)
    # Ultralytics saves logs to [project]/[name]/results.csv
    csv_path = os.path.join(OUTPUT_DIR, "results.csv")

    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip() # Remove spaces

        unified_logs = []
        for _, row in df.iterrows():
            epoch = int(row['epoch'])

            # Map YOLO metrics to our Unified Schema
            # Note: YOLO losses are split (box, seg, cls). We sum them for a 'total' equivalent
            box_loss = row.get('train/box_loss', 0)
            seg_loss = row.get('train/seg_loss', 0)
            cls_loss = row.get('train/cls_loss', 0)

            log_entry = {
                "epoch": epoch,
                "train_loss": float(box_loss + seg_loss + cls_loss),
                "learning_rate": float(row.get('lr/pg0', 0)), # parameter group 0
                "gpu_mem_gb": vram_history.get(epoch + 1, 0.0), # Retrieve from callback

                # Validation Metrics (Mask)
                "val_map_50": float(row.get('metrics/mAP50(M)', 0)),
                "val_map_50_95": float(row.get('metrics/mAP50-95(M)', 0)),
                # YOLO CSV usually doesn't have Map75 explicitly, but Map50-95 covers it.
                # We will leave others as 0.0 to match Schema, or fill if available.
                "val_map_75": 0.0,
                "val_map_small": 0.0,
                "val_map_medium": 0.0,
                "val_map_large": 0.0
            }
            unified_logs.append(log_entry)

        # Save JSON
        json_path = os.path.join(OUTPUT_DIR, "training_log_rich.json")
        with open(json_path, "w") as f:
            json.dump(unified_logs, f, indent=2)
        print(f"Rich logs saved to {json_path}")

        # Smoke Test Validation Logic
        if SMOKE_TEST and len(unified_logs) > 0:
            init_loss = unified_logs[0]['train_loss']
            end_loss = unified_logs[-1]['train_loss']
            validate_smoke_test(OUTPUT_DIR, init_loss, end_loss)

    clear_cuda_cache()

    # 4. Final Benchmark (Test Set)
    print("\n--- Starting Comprehensive Benchmark ---")

    # Run Validation on TEST split
    # Note: Ultralytics doesn't output S/M/L breakdown in the return object easily
    # We focus on mAP and Speed here.
    metrics = model.val(data=DATASET_YAML, split='test', imgsz=IMG_SIZE, batch=1, plots=False)

    # 5. Speed Benchmark (Manual Loop for Fairness)
    # We do NOT rely on `metrics.speed` because it includes pre/post-processing overhead
    # We want to measure the model forward pass exactly like in the Detectron2 script
    print("Benchmarking Speed (Manual Loop)...")

    # Load Image Paths
    with open(DATASET_YAML, 'r') as f:
        data_cfg = yaml.safe_load(f)
    # Construct path logic again (simplify for snippet)
    base_dir = os.path.dirname(DATASET_YAML)
    test_dir_rel = data_cfg.get('test')
    test_images_path = os.path.join(base_dir, test_dir_rel) if not os.path.isabs(test_dir_rel) else test_dir_rel

    # Get images
    img_files = sorted(glob.glob(os.path.join(test_images_path, "*.*")))[:100] # Limit to 100

    times = []
    # Warmup
    if len(img_files) > 0:
        for _ in range(5):
            model.predict(img_files[0], verbose=False)

        for img_f in img_files:
            im = cv2.imread(img_f)
            start = time.time()
            # Run inference only (verbose=False prevents printing)
            _ = model.predict(im, imgsz=IMG_SIZE, verbose=False)
            times.append((time.time() - start) * 1000)

        avg_latency = np.mean(times) if times else 0
        fps = 1000 / avg_latency if avg_latency > 0 else 0
    else:
        avg_latency = 0
        fps = 0

    # 6. Save Final Thesis Results
    final_report = {
        "model": f"YOLO_{RUNNER_NAME}",
        "timestamp": datetime.now().isoformat(),
        "config": {
            "img_size": IMG_SIZE,
            "epochs": epochs,
            "batch_size": batch
        },
        "performance": {
            "latency_ms": avg_latency,
            "fps": fps,
            "mAP_50_95": metrics.seg.map,
            "mAP_50": metrics.seg.map50,
            "mAP_75": metrics.seg.map75,
            # YOLO metrics object doesn't easily expose S/M/L without parsing console output.
            # We set to null or -1 to indicate 'Not Available' in this automated format
            "mAP_Small": -1,
            "mAP_Medium": -1,
            "mAP_Large": -1
        }
    }

    results_path = os.path.join(OUTPUT_DIR, "final_thesis_results.json")
    with open(results_path, "w") as f:
        json.dump(final_report, f, indent=4)

    print(f"Thesis Data Saved to: {results_path}")

    # 7. Visuals
    save_visual_predictions(model, DATASET_YAML, OUTPUT_DIR)

    clear_cuda_cache()

if __name__ == "__main__":
    run_yolo()

Model already exists: yolov12m-seg.pt
Starting Training: YOLOv12_Medium_Tuban
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Colab Notebooks/TA_CuttingRockDescription/datasets/C_2026_1d80_10_10_AUG/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Colab

In [ ]:
def clear_cuda_cache():
    """Clear CUDA GPU memory cache."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()
clear_cuda_cache()
gc.collect()

0